<div style="background:#5D4037; padding:20px; border-radius:10px;">
<h1 style="text-align:center; color:white; margin:0; font-family:'Garamond';">
Proyecto de Café Premium
</h1>
</div>

    Somos el equipo de datos de una marca de café de especialidad que quiere romper el mercado. No aspiramos a ser “otro café más”: nuestro objetivo es convertirnos en la referencia absoluta para personas exigentes que ya no soportan el café plano y repetitivo de las cadenas masivas.
Nuestra misión es clara: convertirnos en el distribuidor número uno para amantes del café premium. No se trata solo de vender más, sino de vender mejor: el origen adecuado, el tueste perfecto y al cliente correcto.
Nuestra propuesta de valor se basa en tres pilares fundamentales:
- seleccionar los mejores orígenes,
- aplicar el tostado más adecuado para cada café,
- ofrecer una calidad alta y consistente.
El reto es que los datos nos llegan en bruto y con bastante caos. Si construimos un modelo deficiente, tomaremos decisiones equivocadas y perderemos ventaja frente a la competencia. Por eso nuestro trabajo consiste en transformar esos datos, darles sentido y convertirlos en una herramienta estratégica que nos permita diferenciarnos de verdad.

## 🚀 Objetivo 1: pasar de una tabla desordenada (`ventas_cafe_raw`) a un modelo relacional en 3FN sin perder informacion.

In [1]:
import sqlite3
import pandas as pd

# Ajusta la ruta si tu ventas-cafe-raw.db está en otra carpeta
con = sqlite3.connect("../01_datos_iniciales/archivos_db/tostador-cafe-raw2.db")

# ¿Qué tablas hay disponibles?
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)

,name
0,ventas_cafe_raw


### 🔎 1. Visualizamos la base de datos inicial y desordenada.

Antes de hacer ningún cambio vamos a realizar unas métricas iniciales para posteriormente verificar que no cometemos errores al realizar el proceso de normalizacion:

---

En primer lugar hacemos una consulta para ver el número de filas que tiene la tabla de datos inicial:

In [2]:
pd.read_sql("SELECT COUNT(*) AS filas_raw FROM 'ventas_cafe_raw'", con)


,filas_raw
0,119


Hacemos una segunda consulta en la que unicamente queremos ver cúantas operaciones de venta hay:

In [3]:
pd.read_sql("""
SELECT COUNT(DISTINCT id_venta) AS ventas_unicas_raw FROM ventas_cafe_raw;
""",con)

,ventas_unicas_raw
0,98


Y una tercera consulta para ver cual es el importe total de ventas:

In [4]:
pd.read_sql("""
SELECT ROUND(SUM(total_venta), 2) AS facturacion_raw FROM ventas_cafe_raw;
""",con)

,facturacion_raw
0,3371.48


In [5]:
from IPython.display import HTML

HTML("""
<style>

/* --- Efecto hover premium con animación --- */
.premium-table td {
    transition: background-color 0.25s ease, color 0.25s ease;
}

.premium-table td:hover {
    background-color: #C8923A !important;
    color: #1A0800 !important;
    cursor: pointer;
}

</style>
""")

In [6]:
# Antes de empezar, importamos las librerias que vamos a utilizar:

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# === Tema de estilo Café Premium para DataFrames ===

premium_styles = [
    {"selector": "th",
     "props": "background-color: #C8923A; color: #1A0800; font-weight: bold; padding: 8px;"},
    
    {"selector": "td",
     "props": "background-color: #2A1205; color: #F2E0C8; padding: 6px;"},
    
    {"selector": "table",
     "props": "border: 1px solid #C8923A; border-collapse: collapse;"}
]

premium_props = {
    "border": "1px solid #C8923A",
    "padding": "6px"
}

# === Zebra premium (filas alternas) ===
def premium_zebra(row):
    if row.name % 2 == 0:
        return ['background-color: #2A1205; color:#F2E0C8'] * len(row)
    else:
        return ['background-color: #1E0D02; color:#F2E0C8'] * len(row)

# === Función para estilizar cualquier DataFrame ===
def estilizar(df, zebra=False):
    styled = (
        df.style
        .set_table_styles(premium_styles)
        .set_properties(**premium_props)
        .set_table_attributes('class="premium-table"')  # Clase para el hover
    )
    if zebra:
        styled = styled.apply(premium_zebra, axis=1)
    return styled

conn = sqlite3.connect("tostador-cafe-raw.db")

<h2><strong>El ejercicio plantea una serie de cuestiones y las divide en varios apartados que resolveremos a continuación:</strong></h2>


<div style="
    background:#2A1205;
    border:1px solid #C8923A;
    border-radius:10px;
    padding:20px;
    color:#F2E0C8;
    font-family:'Lato', sans-serif;
    line-height:1.7;
">

  <h2 style="color:#E8B86D; font-family:'Playfair Display', serif; margin-bottom:10px;">
    Parte A – Diagnóstico
  </h2>

  <h3 style="color:#C8923A; margin-top:18px;">1. Seis problemas de diseño o calidad de dato</h3>
  <ul>
    <li><strong>Redundancia de datos:</strong> <code>email_cliente</code> y <code>nombre_cliente</code> se repiten en cada fila de venta.</li>
    <li><strong>Mezcla de entidades:</strong> la tabla contiene datos del cliente, del producto, de la zona y de la venta en un solo lugar.</li>
    <li><strong>Formato inconsistente:</strong> la columna <code>formato_paquete</code> mezcla unidades (kg, gr) y cantidades.</li>
    <li><strong>Datos subjetivos:</strong> las columnas <code>valoración</code> y <code>comentario_valoración</code> dependen de criterios personales.</li>
    <li><strong>Inconsistencias en el precio del café:</strong> no queda claro si son errores o promociones.</li>
    <li><strong>Valoraciones duplicadas:</strong> la valoración depende del canal (online/tienda), pero lo que se valora es el café.</li>
  </ul>

  <h3 style="color:#C8923A; margin-top:18px;">2. Una anomalía de inserción</h3>
  <p>No se puede dar de alta un café, zona o cliente sin registrar antes una venta, ya que todo depende de <code>id_venta</code>.</p>

  <h3 style="color:#C8923A; margin-top:18px;">3. Una anomalía de actualización</h3>
  <p>Actualizar un email obliga a modificarlo en todas las filas donde aparece, con riesgo de inconsistencias.</p>

  <h3 style="color:#C8923A; margin-top:18px;">4. Una anomalía de borrado</h3>
  <p>Si se borra la única venta de Pola de Lena, desaparece también la zona de la base de datos.</p>

</div>

<div style="
    background:#2A1205;
    border:1px solid #C8923A;
    border-radius:10px;
    padding:20px;
    color:#F2E0C8;
    font-family:'Lato', sans-serif;
    line-height:1.7;
">

  <h2 style="color:#E8B86D; font-family:'Playfair Display', serif; margin-bottom:10px;">
    Parte B – Normalización
  </h2>

  <h3 style="color:#C8923A; margin-top:18px;">1. Propuesta de tablas finales (PK y FK claras)</h3>

  <div style="text-align:center; margin: 1.5rem 0;">
    <img src="../imagenes/propuesta_tablas.png" alt="Diagrama de tablas finales" 
         style="max-width:100%; border:1px solid #C8923A; border-radius:8px;">
  </div>

  <h3 style="color:#C8923A; margin-top:18px;">2. Paso por 1FN, 2FN y 3FN</h3>

  <p>
    Se aplicó el proceso de normalización para eliminar redundancias, dependencias parciales y dependencias transitivas.
    El resultado es un modelo relacional más limpio, eficiente y preparado para análisis posteriores.
  </p>

  <div style="text-align:center; margin: 1.5rem 0;">
    <img src="../imagenes/diagrama_normalizacion.png" alt="Diagrama de normalización" 
         style="max-width:100%; border:1px solid #C8923A; border-radius:8px;">
  </div>

  <h3 style="color:#C8923A; margin-top:18px;">Ejemplo de creación de tablas</h3>

  <div style="
      background:#1E0D02;
      border:1px solid #C8923A;
      border-radius:8px;
      padding:1rem 1.4rem;
      margin-top:1rem;
      font-family:'Playfair Display', serif;
      color:#F2E0C8;
      font-size:0.9rem;
  ">
<pre style="white-space:pre-wrap; color:#F2E0C8;">
CREATE TABLE "cafe" (
  "id_cafe" INTEGER NOT NULL,
  "nombre_cafe" TEXT NOT NULL,
  "origen_cafe" TEXT NOT NULL,
  "proceso_cafe" TEXT NOT NULL,
  "nivel_tueste" TEXT NOT NULL,
  PRIMARY KEY("id_cafe")
);
</pre>
  </div>

  <h3 style="color:#C8923A; margin-top:18px;">Ejemplo de inserción de datos</h3>

  <div style="
      background:#1E0D02;
      border:1px solid #C8923A;
      border-radius:8px;
      padding:1rem 1.4rem;
      margin-top:1rem;
      font-family:'Playfair Display', serif;
      color:#F2E0C8;
      font-size:0.9rem;
  ">
<pre style="white-space:pre-wrap; color:#F2E0C8;">
INSERT INTO cafe (id_cafe, nombre_cafe, origen_cafe, proceso_cafe, nivel_tueste)
SELECT DISTINCT id_cafe, nombre_cafe, origen_cafe, proceso_cafe, nivel_tueste
FROM ventas_cafe_raw;
</pre>
  </div>

  <h3 style="color:#C8923A; margin-top:18px;">Consulta de la tabla resultante</h3>

  <div style="
      background:#1E0D02;
      border:1px solid #C8923A;
      border-radius:8px;
      padding:1rem 1.4rem;
      margin-top:1rem;
      font-family:'Playfair Display', serif;
      color:#F2E0C8;
      font-size:0.9rem;
  ">
<pre style="white-space:pre-wrap; color:#F2E0C8;">
conn = sqlite3.connect("tostador-cafe-raw.db")
pd.read_sql("SELECT * FROM cafe;", conn)
</pre>
  </div>

  <h3 style="color:#C8923A; margin-top:18px;">Resultado de la tabla <code>cafe</code></h3>

  <table style="
      width:100%;
      border-collapse: collapse;
      margin-top:1rem;
      font-size:0.9rem;
      color: #F2E0C8;
      background: #2A1205;
      border:1px solid #C8923A;
      border-radius:8px;
      overflow:hidden;
  ">
    <thead>
      <tr style="background:#C8923A; color:#1A0800;">
        <th style="padding:8px;">id_cafe</th>
        <th style="padding:8px;">nombre_cafe</th>
        <th style="padding:8px;">origen_cafe</th>
        <th style="padding:8px;">proceso_cafe</th>
        <th style="padding:8px;">nivel_tueste</th>
      </tr>
    </thead>
    <tbody>
      <tr><td style="padding:6px;">1</td><td>Colombia Huila</td><td>Colombia</td><td>Lavado</td><td>Medio</td></tr>
      <tr><td style="padding:6px;">2</td><td>Etiopia Yirgacheffe</td><td>Etiopia</td><td>Lavado</td><td>Claro</td></tr>
      <tr><td style="padding:6px;">3</td><td>Brasil Cerrado</td><td>Brasil</td><td>Natural</td><td>Medio</td></tr>
      <tr><td style="padding:6px;">4</td><td>Guatemala Antigua</td><td>Guatemala</td><td>Honey</td><td>Medio-Oscuro</td></tr>
      <tr><td style="padding:6px;">5</td><td>Peru Cajamarca</td><td>Peru</td><td>Lavado</td><td>Claro</td></tr>
      <tr><td style="padding:6px;">6</td><td>Kenia Nyeri</td><td>Kenia</td><td>Lavado</td><td>Claro</td></tr>
    </tbody>
  </table>

</div>

<div style="
    background:#2A1205;
    border:1px solid #C8923A;
    border-radius:10px;
    padding:20px;
    color:#F2E0C8;
    font-family:'Lato', sans-serif;
    line-height:1.7;
">

  <h3 style="color:#C8923A; margin-top:10px;">
    Protocolo anti-pérdida de datos
  </h3>

  <p>
    Una vez pobladas nuestras tablas, comprobamos si las métricas iniciales coinciden con las de la base de datos normalizada.
  </p>

  <h4 style="color:#E8B86D; font-family:'Playfair Display', serif; margin-top:15px;">
    1ª métrica: Número de filas
  </h4>

  <p>El número de filas original era <strong>119</strong>.</p>

</div>

In [12]:

filas = pd.read_sql("SELECT COUNT(*) AS filas_normalizado FROM detalle_venta;", conn)

estilizar(filas)

DatabaseError: Execution failed on sql 'SELECT COUNT(*) AS filas_normalizado FROM detalle_venta;': no such table: detalle_venta

<div style="
    background:#2A1205;
    border:1px solid #C8923A;
    border-radius:10px;
    padding:20px;
    color:#F2E0C8;
    font-family:'Lato', sans-serif;
    line-height:1.7;
">

  <h4 style="color:#E8B86D; font-family:'Playfair Display', serif; margin-top:15px;">
    2ª métrica: Número de ventas unicas
  </h4>

  <p>Antes teniamos <strong>98</strong> ventas únicas.</p>

</div>

In [ ]:
ventas_unicas = pd.read_sql("SELECT COUNT(DISTINCT id_venta) AS ventas_unicas_normalizadas FROM venta;", conn)

estilizar(ventas_unicas)

<div style="
    background:#2A1205;
    border:1px solid #C8923A;
    border-radius:10px;
    padding:20px;
    color:#F2E0C8;
    font-family:'Lato', sans-serif;
    line-height:1.7;
">

  <h4 style="color:#E8B86D; font-family:'Playfair Display', serif; margin-top:15px;">
    3ª métrica: Facturación total
  </h4>

  <p>Nuestro importe final de ventas era de <strong>3371.48</strong>.</p>

</div>

In [ ]:
facturacion = pd.read_sql("SELECT ROUND(SUM(total_venta), 2) AS facturacion_normalizada FROM detalle_venta;", conn)

estilizar(facturacion)

<div style="
    background:#2A1205;
    border:1px solid #C8923A;
    border-radius:10px;
    padding:20px;
    color:#F2E0C8;
    font-family:'Lato', sans-serif;
    line-height:1.7;
    margin-top:20px;
">

  <div style="
      background:#1E0D02;
      border:1px solid #C8923A;
      border-radius:8px;
      padding:1rem 1.4rem;
      font-family:'Playfair Display', serif;
      color:#E8B86D;
      font-size:1.1rem;
      text-align:center;
  ">
    Hemos conseguido pasar de una tabla con datos sin organizar a una tabla normalizada en la que las métricas coinciden.<br>
    <strong style="color:#C8923A;">¡Objetivo 1 CONSEGUIDO!</strong>
  </div>

</div>

## 🚀 Objetivo 2: Responder a las preguntas SQL de negocios

### Consulta 1: Los 5 cafes con mayor facturacion

In [ ]:


cafes = pd.read_sql("""SELECT c.id_cafe, c.nombre_cafe, SUM(dv.total_venta) AS Total_facturacion
FROM detalle_venta AS dv
INNER JOIN cafe as c
	ON c.id_cafe = dv.id_cafe
GROUP BY nombre_cafe
ORDER BY SUM(dv.total_venta) DESC
LIMIT 5;""", conn)


estilizar(cafes, zebra=True)

In [ ]:
plt.bar(cafes['nombre_cafe'], cafes['Total_facturacion'])
plt.xticks(rotation=45)
plt.title("Ventas por café")
plt.xlabel("Café")
plt.ylabel("Total vendido")
plt.show()

### Consulta 2: Zonas con mayor volumen de ventas

Opcion 1. Zonas con mayor volumen de ventas (por facturación)

In [ ]:
zona = pd.read_sql("""SELECT z.id_zona, z.nombre_zona AS Ciudad, SUM(dv.total_venta) AS Ventas
FROM zona as z
INNER JOIN cliente as cl
	ON z.id_zona = cl.id_zona
INNER JOIN venta as v
	ON cl.id_cliente = v.id_cliente
INNER JOIN detalle_venta as dv
	ON v.id_venta = dv.id_venta
GROUP BY nombre_zona
ORDER BY SUM(dv.total_venta) DESC;""", conn)

estilizar(zona, zebra=True)


In [ ]:
plt.pie(zona['Ventas'], labels=zona['Ciudad'], autopct='%1.1f%%')
plt.title("Distribución de ventas por café")
plt.show()

Opcion 2: Zonas con mayor volumen de ventas (por unidades vendidas)

In [ ]:
zona2 = pd.read_sql("""SELECT z.id_zona, z.nombre_zona, SUM(dv.cantidad) AS unidades
FROM zona as z
INNER JOIN detalle_venta as dv
	ON z.id_zona = c.id_zona
INNER JOIN cliente as c
	ON c.id_cliente = v.id_cliente
INNER JOIN venta as v
	ON v.id_venta = dv.id_venta
GROUP BY nombre_zona
ORDER BY SUM(dv.cantidad) DESC;""", conn)


estilizar(zona2, zebra=True)

In [ ]:
plt.bar(zona2['nombre_zona'], zona2['unidades'])
plt.xticks(rotation=45)
plt.title("Ventas por café")
plt.xlabel("Café")
plt.ylabel("Total vendido")
plt.show()

### Consulta 3: Top 3 cafes por valoracion media (minimo 3 valoraciones)

In [ ]:
top3 = pd.read_sql("""SELECT c.nombre_cafe,
		ROUND(AVG(val.valoracion), 2) AS valoracion_media,
		COUNT(*) AS numero_valoraciones
FROM valoraciones AS val
JOIN cafe AS c ON val.id_cafe = c.id_cafe
GROUP BY c.nombre_cafe
HAVING COUNT(*) >= 3
ORDER BY valoracion_media DESC
LIMIT 3;""", conn)

estilizar(top3, zebra=True)

### Consulta 4: Ticket medio por canal ('tienda' vs 'online')

In [ ]:
ticketmedio = pd.read_sql("""SELECT v.canal_venta, 
	ROUND(SUM(dv.cantidad * dv.precio_unitario) / COUNT(DISTINCT v.id_venta), 2)
	AS ticket_medio
FROM venta AS v
JOIN detalle_venta AS dv ON v.id_venta = dv.id_venta
GROUP BY v.canal_venta;""", conn)

estilizar(ticketmedio, zebra=True)

### Consulta 5: recomendacion de cafe para "snobs"

In [ ]:
snobs = pd.read_sql("""SELECT c.nombre_cafe, dv.formato_paquete,
	ROUND(AVG(v.valoracion),2) AS "valoracion media"
FROM detalle_venta dv INNER JOIN cafe c
	ON dv.id_cafe = c.id_cafe
INNER JOIN valoraciones v
	ON c.id_cafe =  v.id_cafe
GROUP BY c.nombre_cafe
ORDER BY "valoracion media" DESC
LIMIT 3;""", conn)

estilizar(snobs, zebra=True)

![podium.png](../imagenes/podium.png)

### Consulta 6: Top 5 de clientes que mas compran

In [ ]:
top_clientes = pd.read_sql("""SELECT cl.id_cliente,
       cl.nombre_cliente,
       SUM(dv.cantidad * dv.precio_unitario) AS total_gastado
FROM cliente AS cl
JOIN venta AS v ON cl.id_cliente = v.id_cliente
JOIN detalle_venta AS dv ON v.id_venta = dv.id_venta
GROUP BY cl.id_cliente, cl.nombre_cliente
ORDER BY total_gastado DESC
LIMIT 5;""", conn)

estilizar(top_clientes, zebra=True)

### Consulta 7: ¿Qué canal tiene más operaciones de venta?

In [ ]:
canal = pd.read_sql("""SELECT 
    cl.nombre_cliente,
    COUNT(DISTINCT CASE WHEN v.canal_venta = 'online' THEN v.id_venta END) AS compras_online,
    COUNT(DISTINCT CASE WHEN v.canal_venta = 'tienda' THEN v.id_venta END) AS compras_tienda,
    CASE 
        WHEN COUNT(DISTINCT CASE WHEN v.canal_venta = 'online' THEN v.id_venta END) = 0 
            THEN 'solo tienda'
        WHEN COUNT(DISTINCT CASE WHEN v.canal_venta = 'tienda' THEN v.id_venta END) = 0 
            THEN 'solo online'
        ELSE 'ambos canales'
    END AS perfil_cliente
FROM cliente cl
JOIN venta v ON cl.id_cliente = v.id_cliente
GROUP BY cl.nombre_cliente
ORDER BY perfil_cliente;""", conn)

estilizar(canal, zebra=True)

In [ ]:
import numpy as np

plt.figure(figsize=(12,6))

x = np.arange(len(canal['nombre_cliente']))   # posiciones en X
width = 0.35                               # ancho de cada barra

plt.bar(x - width/2, canal['compras_online'], width, label='Compras online', color='#6D4C41')
plt.bar(x + width/2, canal['compras_tienda'], width, label='Compras en tienda', color='#A1887F')

plt.xticks(x, canal['nombre_cliente'], rotation=45, ha='right')
plt.ylabel("Número de compras")
plt.title("Comparación de compras por cliente")
plt.legend()
plt.tight_layout()
plt.show()

### Consulta 8: Valoraciones vs proceso_cafe, nivel_tueste

In [ ]:
valoraciones = pd.read_sql("""SELECT 
    c.proceso_cafe, c.nivel_tueste,
    ROUND(AVG(v.valoracion), 2) AS valoracion_media,
    COUNT(v.valoracion) AS total_opiniones
FROM CAFE AS c
INNER JOIN VALORACIONES AS v ON c.id_cafe = v.id_cafe
GROUP BY c.proceso_cafe, c.nivel_tueste
ORDER BY total_opiniones DESC;""", conn)

estilizar(valoraciones, zebra=True)

### Consulta 9: ¿que formato se vende más?

In [ ]:
formato = pd.read_sql("""SELECT 
    formato_paquete, 
    COUNT(*) AS veces_comprado,
    SUM(cantidad) AS unidades_totales,
    ROUND(SUM("total_venta"), 2) AS facturacion_por_formato
FROM DETALLE_VENTA
GROUP BY formato_paquete
ORDER BY unidades_totales DESC;""", conn)

estilizar(formato, zebra=True)

In [ ]:
plt.bar(formato['formato_paquete'], formato['unidades_totales'])
plt.xticks(rotation=45)
plt.title("Ventas por café")
plt.xlabel("Café")
plt.ylabel("Total vendido")
plt.show()